In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/sunithak999/AAI-590-Capstone-group2.git


Cloning into 'AAI-590-Capstone-group2'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 14 (delta 2), reused 10 (delta 2), pack-reused 4 (from 1)
Receiving objects: 100% (14/14), 117.40 MiB | 45.43 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [3]:
!git clone https://github.com/TEJSINGH17/AAI-590-Capstone-group2 json


Cloning into 'json'...
remote: Enumerating objects: 2249, done.
remote: Counting objects: 100% (2249/2249), done.
remote: Compressing objects: 100% (2237/2237), done.
remote: Total 2249 (delta 8), reused 2246 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (2249/2249), 11.43 MiB | 21.27 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [4]:
!pip install ultralytics opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.1 MB/s eta 0:00:00


In [5]:
!pip -q install imageio imageio-ffmpeg

In [ ]:
# =========================================================
# OmniView AI HUD Demo
# Pipeline:
#   1) Reads an MP4 video + per-frame JSON detection files
#   2) Overlays bounding boxes, blind-spot alerts,
#      distance tags, and a top status bar
#   3) Saves annotated frames as a new MP4
#   4) Converts the first N frames to an animated GIF
#   5) Displays the GIF inline in Google Colab
# =========================================================

In [7]:
# =========================================================
# Imports
# =========================================================

import cv2
import json
import os
from glob import glob
import imageio.v2 as imageio
from IPython.display import Image, display


In [8]:
# =========================================================
# PATHS
# =========================================================
VIDEO_PATH = "/content/AAI-590-Capstone-group2/test_data/output_ds_1.mp4"

# One JSON file per frame, written by the YOLO detector
JSON_DIR = "/content/json/runs/hud/ds_1"

OUTPUT_VIDEO_PATH = "/content/omniview_hud_demo.mp4"
OUTPUT_GIF_PATH   = "/content/omniview_hud_demo.gif"

# =========================================================
# SETTINGS
# =========================================================
MAX_FRAMES  = 60    # Cap at 60 frames to keep runtime manageable
CONF_THRESH = 0.40  # Drop anything below 40% confidence
GIF_FRAMES  = 40    # How many frames go into the GIF
GIF_FPS     = 10


In [9]:
# =========================================================
# OPEN VIDEO
# =========================================================
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise ValueError(f"Could not open video: {VIDEO_PATH}")

# Fall back to 30 fps if the container reports 0
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30.0

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Input video: {VIDEO_PATH}")
print(f"Resolution: {width} x {height}")
print(f"FPS: {fps:.2f}")

# =========================================================
# GET JSON FILES
# =========================================================
# Sorted so files are read in frame order
json_files = sorted(glob(os.path.join(JSON_DIR, "*.json")))[:MAX_FRAMES]
if not json_files:
    raise ValueError(f"No JSON files found in: {JSON_DIR}")

print(f"JSON files used: {len(json_files)}")
print(f"Approx processed clip duration: {len(json_files)/fps:.2f} seconds")

Input video: /content/AAI-590-Capstone-group2/test_data/output_ds_1.mp4
Resolution: 1920 x 1080
FPS: 30.00
JSON files used: 60
Approx processed clip duration: 2.00 seconds


In [10]:
# =========================================================
# OUTPUT VIDEO WRITER
# =========================================================
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))
if not out.isOpened():
    raise ValueError("Could not open video writer. Check codec/path.")

# =========================================================
# HUD CONFIG
# =========================================================
vehicle_classes = {"car", "truck", "bus", "motorcycle", "bicycle"}

# Blind-spot zones cover the lower ~52% of the frame, 22% width on each side
left_zone  = (0,               int(height * 0.48), int(width * 0.22), height)
right_zone = (int(width * 0.78), int(height * 0.48), width,           height)


def draw_transparent_box(frame, pt1, pt2, color, alpha=0.18, thickness=2):
    """
    Semi-transparent filled rectangle with a solid border on top.
    alpha controls fill opacity -- 0.18 keeps it visible without blocking the scene.
    """
    overlay = frame.copy()
    cv2.rectangle(overlay, pt1, pt2, color, -1)
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)
    cv2.rectangle(frame, pt1, pt2, color, thickness)


def point_in_zone(cx, cy, zone):
    """Check if a bounding box center falls inside a zone rectangle."""
    x1, y1, x2, y2 = zone
    return x1 <= cx <= x2 and y1 <= cy <= y2


def draw_label(frame, text, x, y, color):
    """
    Text label with a filled background so it stays readable over any scene.
    Clamps y so the label never clips off the top of the frame.
    """
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
    y = max(th + 8, y)
    cv2.rectangle(frame, (x, y - th - 8), (x + tw + 8, y), color, -1)
    cv2.putText(
        frame, text, (x + 4, y - 4),
        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2
    )


def estimate_distance(box_height_px):
    """
    Rough distance estimate based on bounding box height.
    Thresholds tuned for a typical dashcam FOV -- not exact, but good enough
    for a visual warning system.
    """
    if box_height_px >= 140:
        return "CLOSE"
    elif box_height_px >= 70:
        return "MEDIUM"
    else:
        return "FAR"


def safe_box_from_normalized(det, frame_w, frame_h):
    """
    Convert YOLO normalized (x_center, y_center, w, h) to pixel corner coords.
    Clamps to frame bounds so nothing draws outside the image.
    """
    xc = det["x_center"] * frame_w
    yc = det["y_center"] * frame_h
    bw = det["width"]    * frame_w
    bh = det["height"]   * frame_h

    x1 = int(xc - bw / 2)
    y1 = int(yc - bh / 2)
    x2 = int(xc + bw / 2)
    y2 = int(yc + bh / 2)

    x1 = max(0, min(frame_w - 1, x1))
    y1 = max(0, min(frame_h - 1, y1))
    x2 = max(0, min(frame_w - 1, x2))
    y2 = max(0, min(frame_h - 1, y2))

    return x1, y1, x2, y2


In [11]:
# =========================================================
# PROCESS VIDEO + JSON
# =========================================================
processed_frames = 0

for idx, json_path in enumerate(json_files):
    ret, frame = cap.read()
    if not ret:
        print(f"Video ended early at frame index {idx}")
        break

    with open(json_path, "r") as f:
        data = json.load(f)

    detections = data.get("detections", [])

    # Top status bar
    cv2.rectangle(frame, (0, 0), (width, 60), (20, 20, 20), -1)
    cv2.putText(
        frame, "OmniView AI | HUD Prototype", (20, 38),
        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2
    )

    # Blind-spot zone overlays
    draw_transparent_box(frame,
                         (left_zone[0],  left_zone[1]),
                         (left_zone[2],  left_zone[3]),
                         (0, 255, 255))
    draw_transparent_box(frame,
                         (right_zone[0], right_zone[1]),
                         (right_zone[2], right_zone[3]),
                         (0, 255, 255))

    cv2.putText(
        frame, "LEFT BLIND SPOT", (10, left_zone[1] - 10),
        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2
    )
    cv2.putText(
        frame, "RIGHT BLIND SPOT", (width - 210, right_zone[1] - 10),
        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2
    )

    # Reset alerts each frame
    left_alert  = False
    right_alert = False

    for det in detections:
        conf = float(det.get("confidence", 0.0))
        if conf < CONF_THRESH:
            continue

        label = str(det.get("label", "object"))

        try:
            x1, y1, x2, y2 = safe_box_from_normalized(det, width, height)
        except KeyError:
            continue

        if x2 <= x1 or y2 <= y1:
            continue

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2
        box_height   = y2 - y1
        distance_tag = estimate_distance(box_height)

        color = (0, 255, 0)  # Default green; flips red if in a blind spot

        if label.lower() in vehicle_classes:
            if point_in_zone(cx, cy, left_zone):
                color = (0, 0, 255)
                left_alert = True
            elif point_in_zone(cx, cy, right_zone):
                color = (0, 0, 255)
                right_alert = True

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.circle(frame, (cx, cy), 4, color, -1)

        label_text = f"{label} {conf:.2f} | {distance_tag}"
        label_y = y1 - 8 if y1 > 24 else y1 + 22
        draw_label(frame, label_text, x1, label_y, color)

    # Alert banners
    if left_alert:
        cv2.rectangle(frame, (20, 75), (420, 115), (0, 0, 255), -1)
        cv2.putText(
            frame, "ALERT: VEHICLE LEFT", (35, 103),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2
        )

    if right_alert:
        cv2.rectangle(frame, (20, 125), (430, 165), (0, 0, 255), -1)
        cv2.putText(
            frame, "ALERT: VEHICLE RIGHT", (35, 153),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2
        )

    out.write(frame)
    processed_frames += 1

cap.release()
out.release()

print(f"Saved MP4: {OUTPUT_VIDEO_PATH}")
print(f"Frames written: {processed_frames}")
print(f"MP4 exists: {os.path.exists(OUTPUT_VIDEO_PATH)}")
if os.path.exists(OUTPUT_VIDEO_PATH):
    print(f"MP4 size: {os.path.getsize(OUTPUT_VIDEO_PATH)} bytes")


Saved MP4: /content/omniview_hud_demo.mp4
Frames written: 60
MP4 exists: True
MP4 size: 10018523 bytes


In [12]:
# =========================================================
# CONVERT OUTPUT VIDEO TO GIF
# =========================================================
# imageio handles palette-optimized GIFs better than OpenCV
gif_source_frames = []
reader = imageio.get_reader(OUTPUT_VIDEO_PATH)

for i, frame in enumerate(reader):
    if i >= GIF_FRAMES:
        break
    gif_source_frames.append(frame)  # imageio returns RGB

reader.close()

if not gif_source_frames:
    raise ValueError("No frames were read from the output video for GIF conversion.")

imageio.mimsave(OUTPUT_GIF_PATH, gif_source_frames, fps=GIF_FPS)

print(f"Saved GIF: {OUTPUT_GIF_PATH}")
print(f"GIF exists: {os.path.exists(OUTPUT_GIF_PATH)}")
if os.path.exists(OUTPUT_GIF_PATH):
    print(f"GIF size: {os.path.getsize(OUTPUT_GIF_PATH)} bytes")
    print(f"GIF duration: {len(gif_source_frames)/GIF_FPS:.2f} seconds")

Saved GIF: /content/omniview_hud_demo.gif
GIF exists: True
GIF size: 26717304 bytes
GIF duration: 4.00 seconds


In [16]:
# =========================================================
# DISPLAY GIF IN COLAB
# =========================================================
#display(Image(filename=OUTPUT_GIF_PATH))